
# Validate Query Results Against Their Spatio-Temporal Windows

填写下面的 `run_dir` 为一次实验输出目录（包含多个 `query_XX_*` 子文件夹）。运行所有单元格即可：- 生成 `validation_report.csv`（保存在 `run_dir` 目录下）；- 为每个存在违规的查询在其子目录下生成 `out_of_window.csv`；- 打印**前 20 个**违规查询目录名；- 打印**违规查询的总数**。


In [4]:

from pathlib import Path

run_dir = Path(r"D:\implementation\p2psta\implementation\results\run_20250917_134512")

assert run_dir.exists(), f"run_dir not found: {run_dir}"
run_dir


WindowsPath('D:/implementation/p2psta/implementation/results/run_20250917_134512')

In [6]:
import re
from datetime import datetime, timezone
from pathlib import Path
import csv
import json

WINDOW_RE = re.compile(r'^\s*(?P<key>[a-zA-Z_]+)\s*:\s*(?P<val>.+?)\s*$')
RANGE_RE  = re.compile(r'\[(?P<low>[^,]+),\s*(?P<high>[^]]+)\]')  # 提取 [low,high]
BRACKET_CHUNK = re.compile(r'\[.*?\]')

def parse_window_txt(path: Path):
    raw = {}
    with path.open('r', encoding='utf-8') as f:
        content = f.read()

    # 提取所有 key:[...] 结构
    for m in re.finditer(r'(\w+):\[([^\]]+)\]', content):
        key = m.group(1)
        val = f"[{m.group(2)}]"
        raw[key] = val

    # ---- 解析经纬度区间 ----
    for key in ('lat', 'lon'):
        if key not in raw:
            continue
        m = RANGE_RE.search(raw[key])
        if not m:
            raise ValueError(f'{key} 区间格式不对：{raw[key]}')
        raw[f'{key}_min'], raw[f'{key}_max'] = map(float, (m['low'], m['high']))

    # ---- 解析时间区间 ----
    if 'time' in raw:
        m = RANGE_RE.search(raw['time'])
        if not m:
            raise ValueError(f'time 区间格式不对：{raw["time"]}')
        raw['t_start'], raw['t_end'] = m['low'], m['high']

    # ---- 校验必需字段 ----
    required = ['lat_min','lon_min','lat_max','lon_max','t_start','t_end']
    missing = [k for k in required if k not in raw]
    if missing:
        raise ValueError(f"window.txt 缺少字段：{missing} @ {path}")

    return raw

# ---------------- 辅助：把任意格式转 UTC 时间戳 ----------------
def parse_iso_to_epoch_s(s: str):
    s = (s or '').strip()
    if not s:
        return None
    if s.isdigit():
        return int(s)          # 已经是时间戳
    if s.endswith('Z'):
        s = s[:-1] + '+00:00'
    try:
        dt = datetime.fromisoformat(s)
        if dt.tzinfo is None:
            dt = dt.replace(tzinfo=timezone.utc)
        else:
            dt = dt.astimezone(timezone.utc)
        return int(dt.timestamp())
    except Exception:
        for fmt in ['%Y-%m-%d %H:%M:%S','%Y/%m/%d %H:%M:%S','%Y-%m-%d','%Y/%m/%d']:
            try:
                dt = datetime.strptime(s, fmt).replace(tzinfo=timezone.utc)
                return int(dt.timestamp())
            except Exception:
                continue
    return None

In [7]:

# 扫描所有 query_* 子目录
qdirs = sorted([p for p in run_dir.iterdir() if p.is_dir() and p.name.startswith("query_")])
if not qdirs:
    raise RuntimeError(f"No query_* folders found in {run_dir}")

rows = []
violating = []  # 存放有违规的查询（目录名、违规数等）

for qd in qdirs:
    name = qd.name
    window_path = qd / "window.txt"
    results_path = qd / "query_results.csv"
    
    if not window_path.exists():
        rows.append({
            'query_dir': name, 'status': 'missing window.txt',
            'returned_points': 0, 'in_window_points': 0,
            'out_of_window_points': 0, 'out_of_window_ratio': 0.0,
            'lat_min':'','lat_max':'','lon_min':'','lon_max':'','t_start_epoch':'','t_end_epoch':''
        })
        continue
    if not results_path.exists():
        w = parse_window_txt(window_path)
        rows.append({
            'query_dir': name, 'status': 'missing query_results.csv',
            'returned_points': 0, 'in_window_points': 0,
            'out_of_window_points': 0, 'out_of_window_ratio': 0.0,
            'lat_min': w['lat_min'], 'lat_max': w['lat_max'],
            'lon_min': w['lon_min'], 'lon_max': w['lon_max'],
            't_start_epoch': parse_iso_to_epoch_s(w['t_start']),
            't_end_epoch'  : parse_iso_to_epoch_s(w['t_end']),
        })
        continue
    
    # 解析窗口
    w = parse_window_txt(window_path)
    lat_min = float(w['lat_min']); lat_max = float(w['lat_max'])
    lon_min = float(w['lon_min']); lon_max = float(w['lon_max'])
    t_start = parse_iso_to_epoch_s(w['t_start'])
    t_end   = parse_iso_to_epoch_s(w['t_end'])
    
    # 读取结果
    try:
        df = pd.read_csv(results_path)
    except Exception as e:
        rows.append({
            'query_dir': name, 'status': f'failed to read query_results.csv: {e}',
            'returned_points': 0, 'in_window_points': 0,
            'out_of_window_points': 0, 'out_of_window_ratio': 0.0,
            'lat_min': lat_min, 'lat_max': lat_max,
            'lon_min': lon_min, 'lon_max': lon_max,
            't_start_epoch': t_start, 't_end_epoch': t_end,
        })
        continue
    
    expected_cols = ['user','traj_id','lat','lon','datetime']
    missing_cols = [c for c in expected_cols if c not in df.columns]
    if missing_cols:
        returned = len(df)
        out_count = returned
        rows.append({
            'query_dir': name, 'status': f'missing columns: {missing_cols}',
            'returned_points': returned, 'in_window_points': 0,
            'out_of_window_points': out_count,
            'out_of_window_ratio': (out_count/returned if returned>0 else 0.0),
            'lat_min': lat_min, 'lat_max': lat_max,
            'lon_min': lon_min, 'lon_max': lon_max,
            't_start_epoch': t_start, 't_end_epoch': t_end,
        })
        violating.append((name, out_count, returned))
        continue
    
    # 类型转换
    def to_float(x):
        try:
            return float(x)
        except Exception:
            return float('nan')
    df['lat_f'] = df['lat'].apply(to_float)
    df['lon_f'] = df['lon'].apply(to_float)
    df['ts'] = df['datetime'].astype(str).apply(parse_iso_to_epoch_s)
    
    in_lat = (df['lat_f'] >= lat_min) & (df['lat_f'] <= lat_max)
    in_lon = (df['lon_f'] >= lon_min) & (df['lon_f'] <= lon_max)
    in_time = df['ts'].apply(lambda x: (x is not None) and (t_start <= x <= t_end))
    
    in_window = in_lat & in_lon & in_time
    out_window = ~in_window
    
    returned = int(len(df))
    in_count = int(in_window.sum())
    out_count = int(out_window.sum())
    ratio = float(out_count / returned) if returned > 0 else 0.0
    
    # 写违规明细（如果有）
    if out_count > 0:
        vcols = ['user','traj_id','lat','lon','datetime','lat_f','lon_f','ts']
        vdf = df.loc[out_window, vcols].copy()
        vdf.to_csv(qd / "out_of_window.csv", index=False)
        violating.append((name, out_count, returned))
    
    rows.append({
        'query_dir': name, 'status': 'ok',
        'returned_points': returned, 'in_window_points': in_count,
        'out_of_window_points': out_count, 'out_of_window_ratio': ratio,
        'lat_min': lat_min, 'lat_max': lat_max,
        'lon_min': lon_min, 'lon_max': lon_max,
        't_start_epoch': t_start, 't_end_epoch': t_end,
    })

# 保存汇总
report_csv = run_dir / "validation_report.csv"
with report_csv.open('w', newline='', encoding='utf-8') as f:
    w = csv.writer(f)
    w.writerow([
        'query_dir','status','returned_points','in_window_points','out_of_window_points','out_of_window_ratio',
        'lat_min','lat_max','lon_min','lon_max','t_start_epoch','t_end_epoch'
    ])
    for r in rows:
        w.writerow([
            r['query_dir'], r['status'],
            r['returned_points'], r['in_window_points'], r['out_of_window_points'], f"{r['out_of_window_ratio']:.6f}",
            r['lat_min'], r['lat_max'], r['lon_min'], r['lon_max'], r['t_start_epoch'], r['t_end_epoch']
        ])

print(f"Saved summary to: {report_csv}")
len(rows)


Saved summary to: D:\implementation\p2psta\implementation\results\run_20250917_134512\validation_report.csv


2

In [8]:
if 'violating' not in globals() or 'rows' not in globals():
    raise RuntimeError("Run the previous cell first.")

total_queries = len(rows)
violating_sorted = sorted(violating, key=lambda x: (-x[1], x[0]))  # 按违规数降序
top = violating_sorted[:20]

print(f"Total queries: {total_queries}")
print(f"Total violating queries: {len(violating_sorted)}\n")

print("Violating queries (showing up to 20):")
for name, out_cnt, ret_cnt in top:
    ratio = (out_cnt/ret_cnt) if ret_cnt>0 else 0.0
    print(f"  - {name}: {out_cnt}/{ret_cnt} ({ratio:.2%})")



Total queries: 2
Total violating queries: 0

Violating queries (showing up to 20):
